# PrimeKG Graph Exploration

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import config
import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

## Load data

In [2]:
if not config.PRIMEKG_CSV.exists():
    print(f"File not found: {config.PRIMEKG_CSV}")
else:
    df_kg = pd.read_csv(config.PRIMEKG_CSV, low_memory=False)
    print(f"Rows: {len(df_kg):,}, Columns: {len(df_kg.columns)}")
    display(df_kg.head())

Rows: 8,100,498, Columns: 12


,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
0,protein_protein,ppi,0,9796,gene/protein,PHYHIP,NCBI,8889,56992,gene/protein,KIF15,NCBI
1,protein_protein,ppi,1,7918,gene/protein,GPANK1,NCBI,2798,9240,gene/protein,PNMA1,NCBI
2,protein_protein,ppi,2,8233,gene/protein,ZRSR2,NCBI,5646,23548,gene/protein,TTC33,NCBI
3,protein_protein,ppi,3,4899,gene/protein,NRF1,NCBI,11592,11253,gene/protein,MAN1B1,NCBI
4,protein_protein,ppi,4,5297,gene/protein,PI4KA,NCBI,2122,8601,gene/protein,RGS20,NCBI


## Summary and missing values

In [3]:
if 'df_kg' in locals():
    summary = pd.DataFrame({
        'dtype': df_kg.dtypes,
        'null_count': df_kg.isnull().sum(),
        'null_pct': (df_kg.isnull().sum() / len(df_kg) * 100).round(2)
    })
    display(summary)

,dtype,null_count,null_pct
relation,str,0,0.0
display_relation,str,0,0.0
x_index,int64,0,0.0
x_id,str,0,0.0
x_type,str,0,0.0
x_name,str,0,0.0
x_source,str,0,0.0
y_index,int64,0,0.0
y_id,str,0,0.0
y_type,str,0,0.0


## Node types

In [4]:
if 'df_kg' in locals():
    print("Source node types (x_type):")
    print(df_kg['x_type'].value_counts())
    
    print("\nTarget node types (y_type):")
    print(df_kg['y_type'].value_counts())
    
    # Edges where both source and target are disease or phenotype
    mask = (
        df_kg['x_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES) &
        df_kg['y_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES)
    )
    print(f"\nDisease/phenotype edges: {mask.sum():,} ({mask.mean() * 100:.1f}%)")

Source node types (x_type):
x_type
drug                  2805696
gene/protein          2631229
anatomy               1566154
disease                341244
effect/phenotype       257096
biological_process     252202
molecular_function      96723
cellular_component      93102
pathway                 47716
exposure                 9336
Name: count, dtype: int64

Target node types (y_type):
y_type
drug                  2805696
gene/protein          2631229
anatomy               1566154
disease                341244
effect/phenotype       257096
biological_process     252202
molecular_function      96723
cellular_component      93102
pathway                 47716
exposure                 9336
Name: count, dtype: int64



Disease/phenotype edges: 404,880 (5.0%)


## Relation distribution

In [5]:
if 'df_kg' in locals():
    rel_counts = df_kg['relation'].value_counts()
    display(rel_counts.to_frame(name="count"))
    
    print("\nRelation whitelist check:")
    for rel in config.PRIMEKG_RELATION_WHITELIST:
        count = rel_counts.get(rel, 0)
        print(f"  {rel:<20}: {count:,} edges")

,count
relation,
anatomy_protein_present,3036406
drug_drug,2672628
protein_protein,642150
disease_phenotype_positive,300634
bioprocess_protein,289610
cellcomp_protein,166804
disease_protein,160822
molfunc_protein,139060
drug_effect,129568



Relation whitelist check:
  disease_phenotype_positive: 300,634 edges
  disease_disease     : 64,388 edges
  phenotype_phenotype : 37,472 edges
  indication          : 18,776 edges
  contraindication    : 61,350 edges
  disease_phenotype_negative: 2,386 edges


## Keyword search for clinical entities

In [6]:
if 'df_kg' in locals():
    terms = ["sepsis", "pneumonia", "hypotension", "shock", "lactate"]
    
    for term in terms:
        matches = df_kg[
            (df_kg['x_type'].isin(['disease', 'effect/phenotype'])) &
            (df_kg['x_name'].str.contains(term, case=False, na=False))
        ]['x_name'].unique()
        
        print(f"{term} ({len(matches)} matches):")
        for m in matches[:5]:
            print(f"  {m}")
        print()

sepsis (3 matches):
  Sepsis
  bacterial infectious disease with sepsis
  Neonatal sepsis



pneumonia (50 matches):
  Lipoid pneumonia
  Nonspecific interstitial pneumonia
  lymphoid interstitial pneumonia
  desquamative interstitial pneumonia
  cholesterol pneumonia



hypotension (9 matches):
  orthostatic hypotension
  ocular hypotension
  Isolated diastolic hypotension
  Orthostatic hypotension due to autonomic dysfunction
  Postural hypotension with compensatory tachycardia



shock (10 matches):
  Shock
  Anaphylactic shock
  toxic shock syndrome
  streptococcal toxic-shock syndrome
  staphylococcal toxic-shock syndrome



lactate (19 matches):
  Abnormal CSF lactate level
  Increased serum lactate
  Abnormal brain lactate level by MRS
  Abnormal lactate dehydrogenase level
  metabolic myopathy due to lactate transporter defect



## Graph construction and path check

In [7]:
if 'df_kg' in locals():
    # Filter edges by whitelist and node types
    clinical_df = df_kg[
        df_kg['relation'].isin(config.PRIMEKG_RELATION_WHITELIST) &
        df_kg['x_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES) &
        df_kg['y_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES)
    ]
    
    # Build directed graph from dataframe
    G = nx.from_pandas_edgelist(
        clinical_df,
        source="x_name",
        target="y_name",
        edge_attr=["relation"],
        create_using=nx.DiGraph()
    )
    print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    
    # Check path between sample entities
    node_map = {str(n).lower(): n for n in G.nodes()}
    src = next((n for k, n in node_map.items() if "bacterial pneumonia" in k), None)
    tgt = next((n for k, n in node_map.items() if k == "sepsis"), None) or next((n for k, n in node_map.items() if "sepsis" in k), None)
    
    if src and tgt:
        G_undir = G.to_undirected()
        if nx.has_path(G_undir, src, tgt):
            path = nx.shortest_path(G_undir, src, tgt)
            print(f"\nShortest path ({len(path) - 1} hops):")
            for i in range(len(path) - 1):
                u, v = path[i], path[i + 1]
                edge = G.get_edge_data(u, v) or G.get_edge_data(v, u) or {}
                rel = edge.get('relation', 'connected_to')
                print(f"  {u} --[{rel}]--> {v}")
        else:
            print(f"No path found between '{src}' and '{tgt}'")

Graph: 32,341 nodes, 404,523 edges
